In [1]:
!nvidia-smi --query-gpu=name,memory.total,memory.free --format=csv,noheader
!pwd
!ls

NVIDIA GeForce RTX 3050, 6144 MiB, 5249 MiB
/c/Users/RYZEN/vdata/ditto-talkinghead
LICENSE
README.md
core
environment.yaml
example
inference.py
notebook.ipynb
scripts
stream_pipeline_offline.py
stream_pipeline_online.py
venv


In [4]:
import torch
print(torch.__version__)

2.7.1+cpu


In [6]:
!pip install tensorrt librosa tqdm filetype imageio opencv_python_headless scikit-image cython cuda-python imageio-ffmpeg colored polygraphy numpy==2.0.1

  Using cached tensorrt-10.13.2.6.tar.gz (40 kB)
  Preparing metadata (setup.py): started
  Preparing metadata (setup.py): finished with status 'done'
  Using cached librosa-0.11.0-py3-none-any.whl.metadata (8.7 kB)
  Using cached tqdm-4.67.1-py3-none-any.whl.metadata (57 kB)
  Using cached filetype-1.2.0-py2.py3-none-any.whl.metadata (6.5 kB)
  Using cached imageio-2.37.0-py3-none-any.whl.metadata (5.2 kB)
  Using cached opencv_python_headless-4.12.0.88-cp37-abi3-win_amd64.whl.metadata (20 kB)
  Using cached scikit_image-0.25.2-cp311-cp311-win_amd64.whl.metadata (14 kB)
  Using cached cython-3.1.2-cp311-cp311-win_amd64.whl.metadata (6.0 kB)
  Using cached cuda_python-12.9.0-py3-none-any.whl.metadata (4.6 kB)
  Using cached imageio_ffmpeg-0.6.0-py3-none-win_amd64.whl.metadata (1.5 kB)
  Using cached colored-2.3.1-py3-none-any.whl.metadata (3.6 kB)
  Using cached polygraphy-0.49.26-py2.py3-none-any.whl.metadata (5.8 kB)
  Using cached numpy-2.0.1-cp311-cp311-win_amd64.whl.metadata (60 k

  DEPRECATION: Building 'tensorrt' using the legacy setup.py bdist_wheel mechanism, which will be removed in a future version. pip 25.3 will enforce this behaviour change. A possible replacement is to use the standardized build interface by setting the `--use-pep517` option, (possibly combined with `--no-build-isolation`), or adding a `pyproject.toml` file to the source tree of 'tensorrt'. Discussion can be found at https://github.com/pypa/pip/issues/6334
  DEPRECATION: Building 'tensorrt_cu12' using the legacy setup.py bdist_wheel mechanism, which will be removed in a future version. pip 25.3 will enforce this behaviour change. A possible replacement is to use the standardized build interface by setting the `--use-pep517` option, (possibly combined with `--no-build-isolation`), or adding a `pyproject.toml` file to the source tree of 'tensorrt_cu12'. Discussion can be found at https://github.com/pypa/pip/issues/6334


In [3]:
import numpy as np
import torch
import tensorrt as trt
print(np.__version__)
print(torch.__version__)
print(trt.__version__)

2.0.1
2.8.0+cu128
10.13.2.6


In [5]:
# about 1~2min
import os
import torch

def cvt_custom_trt():
    from scripts.cvt_onnx_to_trt import main as cvt_trt
    onnx_dir = "./checkpoints/ditto_onnx"
    trt_dir = "./checkpoints/ditto_trt_custom"
    assert os.path.isdir(onnx_dir)
    os.makedirs(trt_dir, exist_ok=True)
    grid_sample_plugin_file = os.path.join(onnx_dir, "libgrid_sample_3d_plugin.so")
    cvt_trt(onnx_dir, trt_dir, grid_sample_plugin_file)
    return trt_dir


def download_Non_Ampere_trt():
    !pip install --upgrade --no-cache-dir gdown
    !gdown https://drive.google.com/drive/folders/1-1qnqy0D9ICgRh8iNY_22j9ieNRC0-zf?usp=sharing -O ./checkpoints/ditto_trt --folder
    trt_dir = "./checkpoints/ditto_trt"
    return trt_dir


if torch.cuda.get_device_capability()[0] < 8:
    # data_root = cvt_custom_trt()    # cvt
    # The conversion is slow, so you can download pre-converted files.
    data_root = download_Non_Ampere_trt()
else:
    data_root = "./checkpoints/ditto_trt_Ampere_Plus"

In [2]:
import torch
torch.cuda.is_available()

True

In [1]:
# about 1~2min
import os
import torch

def cvt_custom_trt():
    from scripts.cvt_onnx_to_trt import main as cvt_trt
    onnx_dir = "./checkpoints/ditto_onnx"
    trt_dir = "./checkpoints/ditto_trt_custom"
    assert os.path.isdir(onnx_dir)
    os.makedirs(trt_dir, exist_ok=True)
    grid_sample_plugin_file = os.path.join(onnx_dir, "libgrid_sample_3d_plugin.so")
    cvt_trt(onnx_dir, trt_dir, grid_sample_plugin_file)
    return trt_dir


def download_Non_Ampere_trt():
    !pip install --upgrade --no-cache-dir gdown
    !gdown https://drive.google.com/drive/folders/1-1qnqy0D9ICgRh8iNY_22j9ieNRC0-zf?usp=sharing -O ./checkpoints/ditto_trt --folder
    trt_dir = "./checkpoints/ditto_trt"
    return trt_dir


if torch.cuda.get_device_capability()[0] < 8:
    # data_root = cvt_custom_trt()    # cvt
    # The conversion is slow, so you can download pre-converted files.
    data_root = download_Non_Ampere_trt()
else:
    data_root = "./checkpoints/ditto_trt_Ampere_Plus"

In [2]:
# init, about 10s
from inference import StreamSDK, run, stream_run
# data_root = "./checkpoints/ditto_trt_custom"   # model dir
cfg_pkl = "./checkpoints/ditto_cfg/v0.4_hubert_cfg_trt.pkl"     # cfg pkl
print(data_root)
print(cfg_pkl)
SDK = StreamSDK(cfg_pkl, data_root)

./checkpoints/ditto_trt_Ampere_Plus
./checkpoints/ditto_cfg/v0.4_hubert_cfg_trt.pkl
AVATAR REGISTRAR:  {'insightface_det_cfg': {'model_path': './checkpoints/ditto_trt_Ampere_Plus\\insightface_det_fp16.engine', 'device': 'cuda'}, 'landmark106_cfg': {'model_path': './checkpoints/ditto_trt_Ampere_Plus\\landmark106_fp16.engine', 'device': 'cuda'}, 'landmark203_cfg': {'model_path': './checkpoints/ditto_trt_Ampere_Plus\\landmark203_fp16.engine', 'device': 'cuda'}, 'landmark478_cfg': {'blaze_face_model_path': './checkpoints/ditto_trt_Ampere_Plus\\blaze_face_fp16.engine', 'face_mesh_model_path': './checkpoints/ditto_trt_Ampere_Plus\\face_mesh_fp16.engine', 'device': 'cuda'}, 'appearance_extractor_cfg': {'model_path': './checkpoints/ditto_trt_Ampere_Plus\\appearance_extractor_fp16.engine', 'device': 'cuda'}, 'motion_extractor_cfg': {'model_path': './checkpoints/ditto_trt_Ampere_Plus\\motion_extractor_fp32.engine', 'device': 'cuda'}}
OLD PATH:  ./checkpoints/ditto_trt_Ampere_Plus\insightface_det

<frozen importlib._bootstrap_external>:1241: FutureWarning: The cuda.cuda module is deprecated and will be removed in a future release, please switch to use the cuda.bindings.driver module instead.
<frozen importlib._bootstrap_external>:1241: FutureWarning: The cuda.cudart module is deprecated and will be removed in a future release, please switch to use the cuda.bindings.runtime module instead.
<frozen importlib._bootstrap_external>:1241: FutureWarning: The cuda.nvrtc module is deprecated and will be removed in a future release, please switch to use the cuda.bindings.nvrtc module instead.


AssertionError: 

In [5]:
import tensorrt 
tensorrt.__version__

'10.13.2.6'

In [ ]:
pip 